# Hybrid control

In [1]:
import robotic as ry

# 1. Initialize the configuration and load the Panda robot
C = ry.Config()
C.addFile(ry.raiPath('panda/panda.g'))

# Add an obstacle to make it interesting
C.addFrame("obstacle").setPosition([0.3, 0.0, 0.4]).setShape(ry.ST.box, [0.1, 0.4, 0.1]).setColor([0.5, 0.5, 0.5])

# Add a target sphere to reach
C.addFrame("target").setPosition([0.6, 0.2, 0.4]).setShape(ry.ST.sphere, [0.05]).setColor([1, 0, 0])

# 2. Setup KOMO
# 50 time steps, k=2 for acceleration costs
komo = ry.KOMO(C, phases=1, stepsPerPhase=50, k_order=2)

# 3. Define Objectives
# Minimize accelerations to ensure a smooth robot trajectory
komo.addControlObjective(order=2, scale=1.0)

# Collision avoidance: Keep a distance margin from obstacles across all timesteps
komo.addObjective(times=[], 
                  feature=ry.FS.accumulatedCollisions, 
                  frames=[], 
                  type=ry.OT.eq, 
                  scale=[1e2])

# Task objective: the gripper must reach the target exactly at the final time step
komo.addObjective(times=[1.0], 
                  feature=ry.FS.positionDiff, 
                  frames=["panda_gripper", "target"], 
                  type=ry.OT.eq, 
                  scale=[1e2])

# 4. Run the optimizer
komo.optimize()

# 5. Output and display
print("Optimization report:")
print(komo.getReport())

# Play back the generated trajectory in the viewer
komo.view_play(True, 0.1)

ModuleNotFoundError: No module named 'robotic'

In [ ]:
import robotic as ry
import mujoco
import mujoco.viewer
import numpy as np
import time

# =====================================================================
# PART 1: KINEMATIC PLANNING WITH KOMO
# =====================================================================

print("Planning trajectory with KOMO...")

# 1. Initialize the configuration and load the Panda robot
C = ry.Config()
# Note: ry.raiPath requires the 'rai-robotModels' repository to be downloaded
C.addFile(ry.raiPath('panda/panda.g'))

# 2. Add a target frame to reach
C.addFrame("target").setPosition([0.5, 0.3, 0.4]).setShape(ry.ST.sphere, [0.05]).setColor([1, 0, 0])

# 3. Setup KOMO (1 phase, 100 steps, k=2 for accelerations)
steps_per_phase = 100
komo = ry.KOMO(C, phases=1, stepsPerPhase=steps_per_phase, k_order=2)

# Minimize accelerations to ensure a smooth robot trajectory
komo.addControlObjective(order=2, scale=1.0)

# Task objective: the gripper must reach the target exactly at the final time step
komo.addObjective(times=[1.0], 
                  feature=ry.FS.positionDiff, 
                  frames=["panda_gripper", "target"], 
                  type=ry.OT.eq, 
                  scale=[1e2])

# 4. Run the optimizer and extract the trajectory
komo.optimize()
q_traj = komo.getPath()  # Returns a numpy array of shape (100, num_joints)
print(f"KOMO planning complete. Trajectory shape: {q_traj.shape}")


# =====================================================================
# PART 2: PHYSICS SIMULATION WITH MUJOCO
# =====================================================================

print("Executing trajectory in MuJoCo...")

# Note: You need a MuJoCo-compatible Panda XML file with position actuators.
# You can get standard models from the MuJoCo Menagerie: 
# https://github.com/google-deepmind/mujoco_menagerie/tree/main/franka_emika_panda
XML_PATH = "panda_position_control.xml" 

try:
    model = mujoco.MjModel.from_xml_path(XML_PATH)
    data = mujoco.MjData(model)
except ValueError as e:
    print(f"Could not load MuJoCo model. Please ensure you have a valid XML file at '{XML_PATH}'.\nError: {e}")
    exit()

# Set initial state of MuJoCo simulation to match the start of the KOMO trajectory
data.qpos[:7] = q_traj[0][:7]
mujoco.mj_forward(model, data)

# Parameters to map KOMO's discrete steps to MuJoCo's continuous physics timesteps
trajectory_duration = 3.0  # seconds we want the movement to take in simulation
dt = model.opt.timestep
mujoco_steps_per_waypoint = int((trajectory_duration / steps_per_phase) / dt)

# Launch the MuJoCo passive viewer
with mujoco.viewer.launch_passive(model, data) as viewer:
    
    # Give the user a second to look at the initial state before moving
    time.sleep(1.0)
    
    # Iterate through each waypoint planned by KOMO
    for waypoint in q_traj:
        
        # Set the target joint positions for the MuJoCo position actuators
        # (Assuming the first 7 actuators control the 7 DOF arm)
        data.ctrl[:7] = waypoint[:7]
        
        # Step the physics simulation to reach the current waypoint
        for _ in range(mujoco_steps_per_waypoint):
            mujoco.mj_step(model, data)
            
            # Sync the viewer to visualize the current state
            viewer.sync()
            
            # Sleep to match real-time (approximate)
            time.sleep(dt)

print("Simulation complete.")